# Cleaning Downloaded Data from avian-flu

Author: Alexander Maksiaev

Purpose: Clean downloaded data from avian-flu, rename sequences according to convention, de-duplicate from GISAID

In [1]:
# Housekeeping

import os
import glob 
import pandas as pd
import xml.etree.ElementTree as ET
import requests
import time
import numpy as np
import dateutil 
from datetime import datetime
from collections import defaultdict 
import importlib
import utils  
importlib.reload(utils)
from utils import * 

# Make sure you have the correct paths

# home = "C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu_Files/"
home = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu"
# downloads = "C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu_Files/"
downloads = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/"
originals = downloads + "Andersen_Downloads/"
temp_files = downloads + "Andersen_Temp_Files/"
complete_files = downloads + "Andersen_Complete_Files/"

# Day we're updating data
update_date = "04-14-2025"

os.chdir(downloads)

## Read Metadata 

In [2]:
# Read metadata

metadata_folder = originals + "avian-influenza/metadata/"
os.chdir(metadata_folder)

metadata = pd.read_csv("SraRunTable_automated.csv")

# print(len(metadata)) # 7397 rows

# Get rid of missing dates; they won't be counted anyway
for date in metadata["Collection_Date"]:
    if "/" in date or date == "missing":
        metadata = metadata[metadata["Collection_Date"] != date]

# Find only >= 2024 to start
metadata["Collection_Date_Compare"] = metadata["Collection_Date"].apply(lambda x: dateutil.parser.parse(x).strftime("%Y-%m-%d"))
metadata = metadata[metadata["Collection_Date_Compare"] >= datetime(2024, 1, 1).strftime("%Y-%m-%d")] # Note that those with only years will default to today

# Find only >= last date using Release Date from metadata 
metadata["ReleaseDate"] = metadata["ReleaseDate"].apply(lambda x: dateutil.parser.parse(x).strftime("%Y-%m-%d"))
metadata = metadata[metadata["ReleaseDate"] >= datetime(2025, 3, 18).strftime("%Y-%m-%d")]

print(len(metadata)) # 6053 rows between 1/1/2024 and 4/14/2025

991


In [3]:
# # Get list of genotypes

# os.chdir(home)

# genotypes_df = pd.read_excel("genotype_key.xlsx")

# genotypes = list(genotypes_df["Genotype"])

# print(genotypes)

### Naming convention ###
>A/[host]/[geo_loc_name]/[isolate]/[year]|[serotype: H5N1]|[collection_date]|[host_type]|[genotype]

host_type is from manual animal reference

In metadata, we have: host, geo_loc_name, isolate, year

We need: geo_loc_name, collection_date, host_type, genotype

host = Host

geo_loc_name (primary) = geo_loc_name

geo_loc_name (secondary) = genbank_mapping.tsv > genbank_name

isolate = isolate

collection date (primary) = Collection_Date

collection date (secondary) = https://www.ncbi.nlm.nih.gov/genbank/ > BioSample (input: BioSample) > Nucleotide > [first result] > collection_date

serotype = serotype

host type = [from ref] 

genotype = [from genoflu] -- use genoflu_results.tsv

## Get genotype, specific geolocation

In [4]:
# Get genotype from genoflu_results.tsv

os.chdir(metadata_folder)

genoflu_results = pd.read_csv("genoflu_results.tsv", delimiter="\t")

metadata["Genotype"] = genoflu_results["Genotype"]
metadata = metadata[~metadata["Genotype"].str.contains('Not assigned')] # Do not include non-assigned genotypes

# Get only the genotypes we want: B3.13 and D1.1

b313_and_d11_only = genoflu_results[(genoflu_results["Genotype"] == "B3.13") | (genoflu_results["Genotype"] == "D1.1")]
b313_and_d11_only = b313_and_d11_only.rename(columns={"sample": "Run"})
b313_and_d11_only = b313_and_d11_only.drop_duplicates(subset="Run", keep="last")

metadata = metadata.merge(b313_and_d11_only, on=["Run", "Genotype"], how="inner")

print(len(metadata)) 

display(metadata)

303


,Run,Assay Type,AvgSpotLen,Bases,BioProject,BioSample,BioSampleModel,Bytes,Center Name,Collection_Date,...,retraction_detection_date_utc,Collection_Date_Compare,Genotype,date,File Name,"Genotype List Used, >=98.0%",Genotype Sample Title List,Genotype Percent Match List,Genotype Mismatch List,Genotype Average Depth of Coverage List
0,SRR32804537,WGS,148.60,157337234,PRJNA1207547,SAMN47505727,Viral,55371987,USDA-NVSL,2025,...,NaN,2025-04-22,D1.1,2025-04-08_16-17-04,SRR32804537.fa,"PB2:am24, PB1:ea3, PA:am4, HA:ea3, NP:am13, NA...","am24:24-030039-001:PB2, ea3:22-013001-001:PB1,...","99.87%, 99.34%, 99.49%, 99.35%, 99.80%, 98.85%...","3, 15, 9, 11, 3, 11, 1, 7",Ran on FASTA - No Coverage Report
1,SRR32804540,WGS,146.83,125026759,PRJNA1207547,SAMN47505724,Viral,45700705,USDA-NVSL,2025,...,NaN,2025-04-22,D1.1,2025-04-08_16-17-07,SRR32804540.fa,"PB2:am24, PB1:ea3, PA:am4, HA:ea3, NP:am13, NA...","am24:24-030039-001:PB2, ea3:22-013001-001:PB1,...","99.69%, 99.39%, 99.83%, 99.41%, 99.67%, 99.42%...","7, 11, 3, 10, 5, 6, 0, 10",Ran on FASTA - No Coverage Report
2,SRR32804544,WGS,145.13,75849086,PRJNA1207547,SAMN47505721,Viral,27735348,USDA-NVSL,2025,...,NaN,2025-04-22,D1.1,2025-04-08_16-17-10,SRR32804544.fa,"PB2:am24, PB1:ea3, PA:am4, HA:ea3, NP:am13, NA...","am24:24-030039-001:PB2, ea3:22-013001-001:PB1,...","99.69%, 99.50%, 99.60%, 99.65%, 99.60%, 99.05%...","7, 9, 7, 6, 6, 10, 2, 10",Ran on FASTA - No Coverage Report
3,SRR32804545,WGS,148.43,284460923,PRJNA1207547,SAMN47505720,Viral,98766118,USDA-NVSL,2025,...,NaN,2025-04-22,D1.1,2025-04-08_16-17-11,SRR32804545.fa,"PB2:am24, PB1:ea3, PA:am4, HA:ea3, NP:am13, NA...","am24:24-030039-001:PB2, ea3:22-013001-001:PB1,...","99.87%, 99.39%, 99.77%, 99.41%, 99.67%, 99.11%...","3, 11, 5, 10, 5, 10, 0, 9",Ran on FASTA - No Coverage Report
4,SRR32804546,WGS,148.88,213319278,PRJNA1207547,SAMN47505719,Viral,74543339,USDA-NVSL,2025,...,NaN,2025-04-22,D1.1,2025-04-08_16-17-12,SRR32804546.fa,"PB2:am24, PB1:ea3, PA:am4, HA:ea3, NP:am13, NA...","am24:24-030039-001:PB2, ea3:22-013001-001:PB1,...","99.83%, 99.40%, 99.83%, 99.47%, 99.73%, 99.42%...","4, 11, 3, 9, 4, 6, 0, 9",Ran on FASTA - No Coverage Report
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
298,SRR33029747,WGS,147.87,245673050,PRJNA980729,SAMN47843702,Viral,85653750,USDA-NVSL,2025,...,NaN,2025-04-22,D1.1,2025-04-11_06-46-48,SRR33029747.fa,"PB1:ea3, MP:ea3, NA:am4N1, NS:ea3, NP:am13, PA...","ea3:22-013001-001:PB1, ea3:22-013001-001:MP, a...","99.45%, 99.80%, 99.42%, 98.93%, 99.60%, 98.56%...","10, 2, 6, 9, 6, 31, 8, 9",Ran on FASTA - No Coverage Report
299,SRR33029749,WGS,147.88,167615693,PRJNA980729,SAMN47843700,Viral,58998925,USDA-NVSL,2025,...,NaN,2025-04-22,D1.1,2025-04-11_06-46-48,SRR33029749.fa,"PA:am4, NA:am4N1, HA:ea3, NS:ea3, PB1:ea3, MP:...","am4:24-030039-001:PA, am4N1:24-030039-001:NA, ...","98.37%, 99.42%, 99.53%, 98.93%, 99.39%, 99.80%...","35, 6, 8, 9, 11, 2, 9, 6",Ran on FASTA - No Coverage Report
300,SRR33029750,WGS,146.73,95329644,PRJNA980729,SAMN47843699,Viral,33874309,USDA-NVSL,2025,...,NaN,2025-04-22,D1.1,2025-04-11_06-46-48,SRR33029750.fa,"HA:ea3, NA:am4N1, NS:ea3, PB1:ea3, NP:am13, MP...","ea3:22-013001-001:HA, am4N1:24-030039-001:NA, ...","99.53%, 99.42%, 98.93%, 99.44%, 99.60%, 99.80%...","8, 6, 9, 10, 6, 2, 9, 9",Ran on FASTA - No Coverage Report
301,SRR33029751,WGS,146.12,269328456,PRJNA980729,SAMN47843690,Viral,99122459,USDA-NVSL,2025,...,NaN,2025-04-22,D1.1,2025-04-11_06-46-49,SRR33029751.fa,"PA:am4, HA:ea3, NP:am13, MP:ea3, PB2:am24, PB1...","am4:24-030039-001:PA, ea3:22-013001-001:HA, am...","99.30%, 99.06%, 99.73%, 100.00%, 99.74%, 99.12...","15, 16, 4, 0, 6, 20, 10, 7",Ran on FASTA - No Coverage Report


In [ ]:
# # Get specific geolocation from genbank_mapping.tsv

# genbank_mapping = pd.read_csv("genbank_mapping.tsv", delimiter="\t")
# genbank_mapping["Run"] = genbank_mapping["sra_run"]
# genbank_mapping = genbank_mapping.drop_duplicates(subset="Run", keep="first") # Drop duplicates
# genbank_mapping["name_state"] = genbank_mapping["genbank_name"].apply(lambda x: x.split("/")[2]) # Get the name of the state

# print(genbank_mapping)

# metadata_genbank = metadata.merge(genbank_mapping, on=["Run"]) # Only include data that has states

# print(genbank_mapping["name_state"])
# print(len(metadata_genbank))
# display(metadata_genbank) # Maybe there is no state information since 3/18/2025?

                    seg_file  \
0      SRR28752446_HA_cns.fa   
8      SRR28752447_HA_cns.fa   
16     SRR28752448_HA_cns.fa   
24     SRR28752449_HA_cns.fa   
32     SRR28752450_HA_cns.fa   
...                      ...   
38678  SRR32654211_HA_cns.fa   
38686  SRR32654212_HA_cns.fa   
38694  SRR32654213_HA_cns.fa   
38702  SRR32654214_HA_cns.fa   
38710  SRR32654216_HA_cns.fa   

                                            seg_seq_name      sra_run seg  \
0      Consensus_SRR28752446_HA_cns_threshold_0.5_qua...  SRR28752446  HA   
8      Consensus_SRR28752447_HA_cns_threshold_0.5_qua...  SRR28752447  HA   
16     Consensus_SRR28752448_HA_cns_threshold_0.5_qua...  SRR28752448  HA   
24     Consensus_SRR28752449_HA_cns_threshold_0.5_qua...  SRR28752449  HA   
32     Consensus_SRR28752450_HA_cns_threshold_0.5_qua...  SRR28752450  HA   
...                                                  ...          ...  ..   
38678  Consensus_SRR32654211_HA_cns_threshold_0.5_qua...  SRR32654211  HA   

,Run,Assay Type,AvgSpotLen,Bases,BioProject,BioSample,BioSampleModel,Bytes,Center Name,Collection_Date,...,Genotype Mismatch List,Genotype Average Depth of Coverage List,seg_file,seg_seq_name,sra_run,seg,genbank_acc,genbank_seg,genbank_name,name_state


In [ ]:
# If no states

metadata_genbank = metadata

metadata_genbank["name_state"] = "USA"

## Get and save collection date

In [7]:

# Get all dates
metadata_genbank["Collection_Date_Specific"] = metadata_genbank["BioSample"].apply(lambda x: search_collection_date(x, metadata_genbank))

# Save this so we don't have to do it again

os.chdir(temp_files)
metadata_genbank.to_csv("metadata_genbank.csv")

SAMN47505727
Unable to find collection date.
SAMN47505724
Unable to find collection date.
SAMN47505721
Unable to find collection date.
SAMN47505720
Unable to find collection date.
SAMN47505719
Unable to find collection date.
SAMN47505712
Unable to find collection date.
SAMN47505711
Unable to find collection date.
SAMN47505706
Unable to find collection date.
SAMN47505705
Unable to find collection date.
SAMN47505700
Unable to find collection date.
SAMN47505695
Unable to find collection date.
SAMN47505687
Unable to find collection date.
SAMN47505686
Unable to find collection date.
SAMN47505684
Unable to find collection date.
SAMN47505681
Unable to find collection date.
SAMN47505676
Unable to find collection date.
SAMN47505636
Unable to find collection date.
SAMN47505671
Unable to find collection date.
SAMN47505669
Unable to find collection date.
SAMN47505666
Unable to find collection date.
SAMN47505653
Unable to find collection date.
SAMN47505647
Unable to find collection date.
SAMN475057

In [8]:
# # Upload saved data -- if doing this, make sure the above cell is commented out
# os.chdir(temp_files + "saved/")
# metadata_genbank = pd.read_csv("metadata_genbank_4-18-2025.csv")
# os.chdir(temp_files)

# # Get only updated dates

# unknown_dates = metadata_genbank[(metadata_genbank["Collection_Date_Specific"] == "2024") | (metadata_genbank["Collection_Date_Specific"] == "2025")] # Dates we don't have
# known_dates = metadata_genbank[(metadata_genbank["Collection_Date_Specific"] != "2024") & (metadata_genbank["Collection_Date_Specific"] != "2025")] # Dates we've already gotten

# # Get new dates also 
# # new_dates = metadata_genbank["BioSample"].apply(lambda x: search_collection_date(x, metadata_genbank) if )

# updated_unknown_dates = unknown_dates["BioSample"].apply(lambda x: search_collection_date(x, unknown_dates)) # Update unknown dates, if possible

# metadata_genbank = pd.concat([known_dates, unknown_dates], ignore_index=True, sort=True)

# # Remove pre-2024 dates

# metadata_genbank["Collection_Date_Compare"] = metadata_genbank["Collection_Date_Specific"].apply(lambda x: dateutil.parser.parse(x).strftime("%Y-%m-%d"))
# metadata_genbank = metadata_genbank[metadata_genbank["Collection_Date_Compare"] >= datetime(2024, 1, 1).strftime("%Y-%m-%d")]

# print(metadata_genbank[["Collection_Date_Specific"]])

# display(metadata_genbank)

## Get host type

In [9]:
# Create animals ref if needed

unique_animals_all = sort_animals_andersen(metadata_genbank)

# Flatten unique_animals_all
every_unique_animal = []
for animal in unique_animals_all:
    every_unique_animal.append(animal)

print(every_unique_animal)

unique_animals_set = list(set(every_unique_animal)) # Get rid of duplicates

os.chdir(downloads)

animals_ref = pd.read_csv("animals_ref.csv") # Upload animals ref

# If animal not in ref1, put in ref2

common_animals = []
# Check if animals in unique_animals_set are in ref1
for animal in unique_animals_set:
    for col in animals_ref.columns:
        if animal in animals_ref[col].values and type(animal) == str:
            common_animals.append(animal)

# If not in ref1, make a list of the new animals
different_animals = []
for animal in unique_animals_set:
    if animal not in common_animals:
        different_animals.append(animal)

print(different_animals)

# Add to dataframe
animals_df = animals_ref
# Make different_animals same length as dataframe, if shorter
if len(different_animals) < len(animals_df):
    number_of_times_to_add_nan = len(animals_df) - len(different_animals)
    for i in range(number_of_times_to_add_nan):
        different_animals.append(float('nan'))
# If longer, deal with that later

animals_df["new"] = (different_animals)

print(animals_df)

animals_df.to_csv("animals_ref_to_sort.csv") # Make sure name is different to avoid overwriting the first reference 


['great horned owl', 'bald eagle', 'guineafowl', 'vulture', 'herring gull', 'cattle', 'falco peregrinus', 'red-shouldered hawk', 'black vulture', 'barn owl', 'falcon', 'sparrow', 'cackling goose', 'duck', 'merganser', 'cat', 'chicken', 'pet food', 'dolphin', 'quail', 'mallard', 'bufflehead', 'common eider', 'goose', 'american black duck', "cooper's hawk", 'common raven', 'bear', 'great black-backed gull', 'turkey', 'turkey vulture', 'skunk', 'snow goose', 'trumpeter swan', 'red-breasted merganser', 'canada goose', 'swan', 'red-tailed hawk', 'common merganser', 'snowy owl']
['falco peregrinus', 'common eider', 'great black-backed gull', 'common merganser']
                 avian               cattle        feline   other_mammal  \
0     great_horned_owl            dairy_cow           cat         bobcat   
1         common_raven               cattle  domestic_cat    house_mouse   
2        cooper's_hawk  cattle milk product     feral_cat          skunk   
3         coopers_hawk          

In [10]:
# Get animals from animal reference
os.chdir(downloads)
animals_ref = pd.read_csv("animals_ref.csv")
fix_animals_andersen(metadata_genbank, animals_ref) # Get host type

metadata_genbank["years"] = metadata_genbank["Collection_Date"].apply(lambda x: str(x).split("-")[0]) # Get year only from collection date

## Make names using all the attributes we collected

In [11]:
for num, collection_date in enumerate(metadata_genbank["Collection_Date_Specific"]):
    if collection_date != collection_date: # If nan
        metadata_genbank.loc[num, "Collection_Date_Specific"] = metadata_genbank.loc[num, "years"]
    else: # If actual date
        if len(str(collection_date)) == 4: # If it's a year
            # print("caught")
            metadata_genbank.loc[num, "Collection_Date_Specific"] = collection_date
        else:
            parsed_date = dateutil.parser.parse(collection_date)
            date = parsed_date.strftime("%Y-%m-%d") # Make sure it doesn't default to today, if just a year
            metadata_genbank.loc[num, "Collection_Date_Specific"] = date

# Make names

names = ">A/" + metadata_genbank["Host"] + "/" + metadata_genbank["name_state"] + "/" + metadata_genbank["isolate"] + "/" + metadata_genbank["years"].apply(lambda x: str(x)) + "|H5N1|" + metadata_genbank["Collection_Date_Specific"].apply(lambda x: str(x)) + "|" + metadata_genbank["Host_Type"] + "|" + metadata_genbank["Genotype"]

metadata_genbank["Name"] = names

# metadata_genbank.to_csv("metadata_genbank_named.csv")

# display(metadata_genbank)

In [12]:
print(metadata_genbank)

             Run Assay Type  AvgSpotLen      Bases    BioProject  \
0    SRR32804537        WGS      148.60  157337234  PRJNA1207547   
1    SRR32804540        WGS      146.83  125026759  PRJNA1207547   
2    SRR32804544        WGS      145.13   75849086  PRJNA1207547   
3    SRR32804545        WGS      148.43  284460923  PRJNA1207547   
4    SRR32804546        WGS      148.88  213319278  PRJNA1207547   
..           ...        ...         ...        ...           ...   
298  SRR33029747        WGS      147.87  245673050   PRJNA980729   
299  SRR33029749        WGS      147.88  167615693   PRJNA980729   
300  SRR33029750        WGS      146.73   95329644   PRJNA980729   
301  SRR33029751        WGS      146.12  269328456   PRJNA980729   
302  SRR33029752        WGS      145.33  270721593   PRJNA980729   

        BioSample BioSampleModel     Bytes Center Name Collection_Date  ...  \
0    SAMN47505727          Viral  55371987   USDA-NVSL            2025  ...   
1    SAMN47505724        

## Make FASTA files

In [13]:
# Get information to create the fasta files

fasta_folder = originals + "avian-influenza/fasta/"

os.chdir(fasta_folder)

segments = ["PB2", "PB1", "PA", "NS", "NP", "NA", "MP", "HA"]
pairs = []
fasta_files = {}

for genotype in ["B3.13", "D1.1"]:
    for segment in segments:
        pair = genotype + "_" + segment
        pairs.append(pair)

for pair in pairs:
    fasta_files[pair] = [] # List to hold fasta files

for run in metadata_genbank["Run"].values: # For each run 
    for dirpath, dirs, files in os.walk(fasta_folder): # Find the fasta file
        for file in files:
            file_name = os.path.join(dirpath, file) # Get file name
            # print(file_name)
            if run in file_name: # Note that there will be ~8 files total with that run name
                # Make a fasta file and put it in the list
                with open(file_name) as f:
                    lines = f.readlines()
                    sequence = lines[1] 
                    # Each run/segment pair has one sequence -- it's placed into a file with other run/segment pairs with the same segment and genotype
                    header = metadata_genbank[metadata_genbank["Run"] == run].loc[:, "Name"].values[0]
                    genotype = metadata_genbank[metadata_genbank["Run"] == run].loc[:, "Genotype"].values[0]
                    # print(header)
                    # print(genotype)
                    # break 
                    segment = file_name.split("_")[-2]
                    # Find the pair that corresponds to 
                    pair_name = genotype + "_" + segment
                    this_specific_fasta = []
                    for pair in pairs:
                        # print(pair)
                        # print(pair_name)
                        if pair_name == pair:
                            this_specific_fasta.append(header)
                            this_specific_fasta.append(sequence)
                            fasta_files[pair].append(this_specific_fasta)
                f.close()
        break 

In [14]:
# Create fasta files 

os.chdir(temp_files)

for pair in fasta_files.keys():
    output_path = temp_files + pair + "_andersen_" + update_date + ".fasta" 

    output_file = open(output_path, "w")
    for item in fasta_files[pair]:
        # for item in item:
        # item = fasta_files[pair]
        try:
            name = str(item[0].values[0]) # See if this is one we didn't have a collection date for
        except:
            name = str(item[0])
        print(name)
        # First is header, second is sequence
        # print(value)
        output_file.write(name + "\n")
        output_file.write(item[1])
    output_file.close()

>A/CAT/unknown/25-005779-001/2025|H5N1|2025|feline|B3.13
>A/CATTLE/unknown/25-007943-001/2025|H5N1|2025|cattle|B3.13
>A/CATTLE/unknown/25-007915-001/2025|H5N1|2025|cattle|B3.13
>A/CATTLE/unknown/25-007201-004/2025|H5N1|2025|cattle|B3.13
>A/CATTLE/unknown/25-007185-006/2025|H5N1|2025|cattle|B3.13
>A/TURKEY/unknown/25-007640-001/2025|H5N1|2025|avian|B3.13
>A/PET FOOD/unknown/25-007637-012/2025|H5N1|2025|other|B3.13
>A/PET FOOD/unknown/25-007637-011/2025|H5N1|2025|other|B3.13
>A/PET FOOD/unknown/25-007637-010/2025|H5N1|2025|other|B3.13
>A/PET FOOD/unknown/25-007637-007/2025|H5N1|2025|other|B3.13
>A/PET FOOD/unknown/25-007637-006/2025|H5N1|2025|other|B3.13
>A/PET FOOD/unknown/25-007637-003/2025|H5N1|2025|other|B3.13
>A/PET FOOD/unknown/25-005370-008/2025|H5N1|2025|other|B3.13
>A/PET FOOD/unknown/25-005370-006/2025|H5N1|2025|other|B3.13
>A/PET FOOD/unknown/25-005370-004/2025|H5N1|2025|other|B3.13
>A/PET FOOD/unknown/25-005370-002/2025|H5N1|2025|other|B3.13
>A/PET FOOD/unknown/25-005350-004/

## De-Duplication

In [21]:
# De-duplication 

# Gisaid 

# gisaid1 = downloads + "GISAID_Complete_Fasta_Files/01-01-2024--03-31-2025/"
# gisaid2 = downloads + "GISAID_Complete_Fasta_Files/04-01-2025--04-14-2025/"
gisaid = downloads + "GISAID_Complete_Fasta_Files/03-19-2025--04-14-2025/"

os.chdir(gisaid)

dfs_gisaid = create_dataframes(gisaid)
# dfs_gisaid2 = create_dataframes(gisaid2)

B3.13_HA
B3.13_MP
B3.13_NA
B3.13_NP
B3.13_NS
B3.13_PA
B3.13_PB1
B3.13_PB2
D1.1_HA
D1.1_MP
D1.1_NA
D1.1_NP
D1.1_NS
D1.1_PA
D1.1_PB1
D1.1_PB2


In [22]:
# for key in dfs_gisaid.keys():
#     dataframes = dfs_gisaid[key]
#     print(dataframes)

# print(dfs_gisaid.keys())

In [23]:
# Do the same with Andersen 

dfs_andersen = create_dataframes(temp_files)

B3.13_HA
B3.13_HA
B3.13_MP
B3.13_MP
B3.13_NA
B3.13_NA
B3.13_NP
B3.13_NP
B3.13_NS
B3.13_NS
B3.13_PA
B3.13_PA
B3.13_PB1
B3.13_PB1
B3.13_PB2
B3.13_PB2
D1.1_HA
D1.1_HA
D1.1_MP
D1.1_MP
D1.1_NA
D1.1_NA
D1.1_NP
D1.1_NP
D1.1_NS
D1.1_NS
D1.1_PA
D1.1_PA
D1.1_PB1
D1.1_PB1
D1.1_PB2
D1.1_PB2


In [24]:
for key in dfs_andersen.keys():
    dataframes = dfs_andersen[key]
    print(key)

B3.13_HA
B3.13_MP
B3.13_NA
B3.13_NP
B3.13_NS
B3.13_PA
B3.13_PB1
B3.13_PB2
D1.1_HA
D1.1_MP
D1.1_NA
D1.1_NP
D1.1_NS
D1.1_PA
D1.1_PB1
D1.1_PB2


In [26]:
# Merge dataframes and drop duplicates

full_dfs = defaultdict(list)

for i, andersen_key in enumerate(dfs_andersen.keys()):
    gisaid_key = list(dfs_gisaid.keys())[i]
    # gisaid2_key = list(dfs_gisaid2.keys())[i]

    andersen_df = dfs_andersen[andersen_key][0]
    gisaid_df = dfs_gisaid[gisaid_key][0]
    # gisaid2_df = dfs_gisaid2[gisaid2_key][0]

    full_df = pd.concat([andersen_df, gisaid_df], ignore_index=True)
    full_df = full_df.drop_duplicates(subset="isolate_partial", keep="last")
    full_dfs[andersen_key].append(full_df)

# for andersen_key in dfs_andersen.keys():
#     # print("Andersen: ", andersen_key)
#     for gisaid_key in dfs_gisaid.keys():
#         # print("GISAID: ", gisaid_key)
#         if andersen_key == gisaid_key:
#             print(gisaid_key)
#             andersen_df = dfs_andersen[key][0]
#             # print(andersen_df)
#             gisaid_df = dfs_gisaid[key][0]
#             full_df = andersen_df.merge(gisaid_df, how = "outer")
#             # print(full_df)
#             full_df = full_df.drop_duplicates(subset=["isolate_partial"], keep="last")
#             full_dfs[andersen_key].append(full_df)

# print(full_dfs.keys())

# for key in dfs_andersen.keys():
#     dataframes = dfs_andersen[key]
#     for i, df in enumerate(dataframes):
#         print(i)
#         try:
#             full_df = df.merge(dfs_gisaid[key][i], how="outer")
#             # print(full_df)
#             full_df = full_df.drop_duplicates(subset=["isolate_partial"])
#             full_dfs[key].append(full_df)
#         except:
#             print("Failed to merge dataframes in ", key)

print(full_dfs)



defaultdict(<class 'list'>, {'B3.13_HA': [    isolate_partial                                        full_header  \
0        005779-001  >A/CAT/unknown/25-005779-001/2025|H5N1|2025|fe...   
1        007943-001  >A/CATTLE/unknown/25-007943-001/2025|H5N1|2025...   
3        007201-004  >A/CATTLE/unknown/25-007201-004/2025|H5N1|2025...   
4        007185-006  >A/CATTLE/unknown/25-007185-006/2025|H5N1|2025...   
9        007637-007  >A/PET FOOD/unknown/25-007637-007/2025|H5N1|20...   
..              ...                                                ...   
384      000600-002  >A/dairy_cow/California/25_000600-002/2024|H5N...   
385      000590-002  >A/dairy_cow/California/25_000590-002/2024|H5N...   
386      005209-001  >A/dairy_cow/California/25_005209-001/2024|H5N...   
387      004920-005  >A/dairy_cow/California/25_004920-005/2024|H5N...   
388      004920-004  >A/dairy_cow/California/25_004920-004/2024|H5N...   

                                              sequence  
0    ATGAAGA

In [27]:
# If none in one database, only use the other and drop duplicates

# full_dfs = defaultdict(list)
# for key in dfs_andersen.keys():
#     print(key)
# # for key in ["D1.3"]:
#     dataframes = dfs_andersen[key]
#     for i, df in enumerate(dataframes):
#         print(i)
#         try:
#             # full_df = df.merge(dfs_gisaid[key][i], how="outer")
#             # print(full_df)
#             full_df = full_df.drop_duplicates(subset=["isolate_partial"])
#             full_dfs[key].append(full_df)
#         except:
#             print("Failed to merge dataframes in ", key)

## Create FASTA files combining Andersen and GISAID

In [28]:
# Create FASTA files per segment

combined_files = downloads + "Combined_Files/"

os.chdir(combined_files)
for pair in full_dfs.keys():
    print(pair)
    output_path = combined_files + pair + "_combined_" + update_date + ".fasta" 

    output_file = open(output_path, "w")
    for item in full_dfs[pair]:
        # for item in item:
        # item = fasta_files[pair]
        for index, row in item.iterrows():
            name = item.loc[index, "full_header"]
            sequence = item.loc[index, "sequence"]
            # print(name)
        # First is header, second is sequence
        # print(value)
            output_file.write(name)
            output_file.write(sequence)
    output_file.close()

B3.13_HA
B3.13_MP
B3.13_NA
B3.13_NP
B3.13_NS
B3.13_PA
B3.13_PB1
B3.13_PB2
D1.1_HA
D1.1_MP
D1.1_NA
D1.1_NP
D1.1_NS
D1.1_PA
D1.1_PB1
D1.1_PB2


In [21]:
# os.chdir(complete_files)

# segment_files = []
# for dirpath, dirs, files in os.walk(temp_files): # + "01-01-2024--03-31-2025_renamed/"):
#     for segment in ["PB2", "PB1", "PA", "NS", "NP", "NA", "MP", "HA"]:
#         base = []
#         for file in files:
#             file_name = os.path.join(dirpath, file)
#             # print(file_name)
#             if segment in file_name:
#                 with open(file_name) as f:
#                     lines = f.readlines()
#                     for line in lines:
#                         base.append(line)
#                 f.close()
#         os.chdir(complete_files)
#         output_path = complete_files + segment + "_andersen_" + update_date + ".fasta" # Genotype and Segment should all be the same
#         output_file = open(output_path, "w")
#         for b in base:
#             output_file.write(b)
#         output_file.close()
#     break 